# Tahap 2 - Data Understanding

**Tujuan:** mengenal dataset sebelum dibersihkan: ukuran, tipe data, missing values, rentang waktu, duplikat, dan pemetaan kolom ke 8 pertanyaan analisis.

Dataset yang dipakai: **TikTok Video Metadata** (`tiktok_data.csv`) dari Kaggle, diunduh manual (metode B).

In [ ]:
# Sel 1: Muat data dengan aman
import os
import pandas as pd

BASE = '/content'
PATH = os.path.join(BASE, 'data', 'tiktok_data.csv')

# id_sound dibaca sebagai teks: ID 19 digit akan kehilangan presisi kalau dibaca sebagai angka.
df = pd.read_csv(PATH, encoding='utf-8-sig', dtype={'id_sound': 'string'})

print('nama file   :', os.path.basename(PATH))
print('ukuran      :', round(os.path.getsize(PATH) / 1024, 1), 'KB')
print('baris x kolom:', df.shape)
print()
print(df.dtypes)

In [ ]:
# Sel 2: Inventaris kolom, lalu simpan tabel ringkasan
os.makedirs(os.path.join(BASE, 'outputs'), exist_ok=True)

inventaris = pd.DataFrame({
    'kolom': df.columns,
    'tipe_data': df.dtypes.astype(str).values,
    'non_null': df.notna().sum().values,
    'missing': df.isna().sum().values,
    'persen_missing': (df.isna().mean() * 100).round(1).values,
    'nilai_unik': df.nunique(dropna=True).values,
})
print(inventaris.to_string(index=False))
inventaris.to_csv(os.path.join(BASE, 'outputs', '02_inventaris_kolom.csv'),
                  index=False, encoding='utf-8-sig')

In [ ]:
# Sel 3: Duplikat + statistik numerik
# Median dipakai karena distribusi views/likes sangat condong ke kanan.
print('baris identik penuh:', int(df.duplicated().sum()))
print('id_video duplikat  :', int(df['id_video'].duplicated().sum()))
print()

num = ['duration_seconds', 'author_followers', 'author_likes', 'views', 'likes']
stat = df[num].agg(['min', lambda s: s.quantile(.25), 'median',
                    lambda s: s.quantile(.75), 'mean', 'max']).T
stat.columns = ['min', 'q25', 'median', 'q75', 'mean', 'max']
print(stat.round(1).to_string())
print()
print('views == 0        :', int((df['views'] == 0).sum()))
print('likes > views     :', int((df['likes'] > df['views']).sum()))
print('durasi == 0 detik :', int((df['duration_seconds'] == 0).sum()))

In [ ]:
# Sel 4: Rentang tanggal dari human_time
wt = pd.to_datetime(df['human_time'], errors='coerce')
print('berhasil di-parse:', int(wt.notna().sum()), '| gagal:', int(wt.isna().sum()))
print('rentang          :', wt.min(), 's/d', wt.max())
print('tahun unik      :', sorted(wt.dt.year.dropna().astype(int).unique()))
print('timestamp dipakai >1 video:', int((wt.value_counts() > 1).sum()))

In [ ]:
# Sel 5: Cek kualitas & privasi (tanpa menampilkan teks asli yang memuat akun)
inti = ['duration_seconds', 'human_time', 'author_followers', 'id_sound', 'views', 'likes']
print('baris dengan semua kolom inti kosong:', int(df[inti].isna().all(axis=1).sum()))
print()

salah = pd.read_csv(PATH, encoding='utf-8-sig')['id_sound'].dropna().iloc[0]
benar = df['id_sound'].dropna().iloc[0]
print('id_sound dibaca default :', salah)
print('id_sound dibaca sbg teks:', benar)
print()

cap = df['text_part'].fillna('')
print('caption memuat @ (potensi akun pribadi):', int(cap.str.contains('@', regex=False).sum()))
print('caption memuat tautan                  :', int(cap.str.contains(r'https?://|www\.', regex=True).sum()))
print('id_video berurutan 1..N               :',
      bool((df['id_video'].sort_values().to_numpy() == range(1, len(df) + 1)).all()))

In [ ]:
# Sel 6: Catat asal data secara manual (karena tidak pakai Kaggle API)
prov = pd.DataFrame([{
    'nama_file': 'tiktok_data.csv',
    'judul_dataset': 'TikTok Video Metadata',
    'url_kaggle': 'https://www.kaggle.com/datasets/maratsaratov/tiktok-data',
    'lisensi': 'Apache 2.0',
    'cara_pengumpulan': 'Tidak dijelaskan rinci di halaman Kaggle; hanya disebut metadata video TikTok nyata dari berbagai topik dan wilayah',
    'rentang_tanggal': '2015-07-10 s/d 2025-06-14 (human_time yang berhasil di-parse)',
    'catatan': 'Snapshot; tanpa komentar, share, bahasa, wilayah, dan usia penonton',
}])
prov.to_csv(os.path.join(BASE, 'outputs', 'provenance.csv'),
            index=False, encoding='utf-8-sig')
print(prov.T.to_string(header=False))

## Pemetaan kolom ke 8 pertanyaan

| # | Pertanyaan | Kolom yang dipakai | Status |
|---|---|---|---|
 | 1 | Jenis konten dengan views/likes tertinggi | `hashtags`, `text_part`, `views`, `likes` | **Sebagian** — hanya views & likes; tidak ada comments/shares; tidak ada kolom kategori resmi |
 | 2 | Durasi vs tingkat interaksi | `duration_seconds`, `views`, `likes` | **Sebagian** — `engagement_rate` tidak bisa dihitung; pakai `like_rate = likes/views` sebagai proxy |
 | 3 | Jam & hari posting | `human_time` | **Bisa** |
 | 4 | Hashtag, sound, kata kunci caption | `hashtags`, `id_sound`, `text_part` | **Bisa** |
 | 5 | Follower vs engagement rate | `author_followers`, `views`, `likes` | **Sebagian** — hanya like rate |
 | 6 | Perbedaan bahasa/wilayah, fokus Indonesia | tidak ada | **Tidak bisa** — tidak ada kolom bahasa, negara, atau region |
 | 7 | Jenis interaksi dominan per konten | tidak ada | **Tidak bisa** — tidak ada `comments` dan `shares` |
 | 8 | Rekomendasi untuk kreator/brand | seluruh temuan di atas | **Bisa**, dengan keterbatasan jelas |

Catatan penting: deskripsi Kaggle menyebut data berasal dari "berbagai topik dan wilayah", tetapi **tidak ada kolom wilayah** di file. Jadi kita tidak bisa mengklaim analisis Indonesia.

## Yang sudah diketahui dari eksekusi

- File `tiktok_data.csv`, 141,0 KB, **760 baris × 10 kolom**.
- Tipe data: `id_video` int64; `duration_seconds`, `author_followers`, `author_likes`, `views`, `likes` float64; `text_part`, `hashtags`, `human_time` teks; `id_sound` sudah dibaca sebagai teks.
- `human_time` berhasil di-parse untuk 751 baris, rentang **2015-07-10 sampai 2025-06-14**.
- Tidak ada baris identik dan tidak ada `id_video` duplikat.
- Median views 394.800, tapi mean 5.661.243 — ini bukti distribusi sangat condong ke kanan, makanya median yang dipakai.
- 9 baris memiliki semua kolom inti kosong; 16 baris durasi = 0 detik. Keduanya perlu diputuskan di Tahap 3.
- `id_sound` akan rusak (hilang digit) jika dibaca default → sudah diperbaiki dengan `dtype='string'`.
- 63 caption memuat tanda `@` → berpotensi memuat akun pribadi; akan ditutup di Tahap 3 sebelum data disimpan.

**Lanjut:** Tahap 3 - Data Cleaning. Setelah itu baru EDA.